# Post-processing

1) [Absurd values removal](#absurd-values-removal)
2) [Save results to gpkg](#save-results-to-gpkg)

In [2]:
# dependencies
import os
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.colors as mcolors
import pickle
from tqdm import tqdm
import open3d as o3d
import laspy
from plyfile import PlyData, PlyElement
import rasterio
from rasterio.transform import from_origin
from copy import deepcopy

import geopandas as gpd
from shapely.geometry import Point
import pandas as pd
from time import time

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


### Absurd values removal

Load the geopackage output and find absurde values. Create then a status code as followed:
- 0: correct value
- 1: absurd value
- 2: correct value but sibling to an absurd value

In [3]:
def get_translate_from_element(el):
    bbox_dict = el[1][0]
    transform = el[1][3]

    xmin, ymin, zmin = bbox_dict['min_bound']
    xmax, ymax, zmax = bbox_dict['max_bound']
    center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
    translated = np.linalg.matmul(transform, center)
    norm = float(np.linalg.norm(translated - center))

    return norm

In [4]:
# laoding of gpkg file
# src_gpkg = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_5_min_points\3000_max_lvl_7\results\points_translate.gpkg"
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_6_neigh_on_target\test_speed_up_to_lvl_8\results\pyramid_transforms_test.pickle"
src_results = src_transforms.split('.pickle')[0] + "_with_absurd_vals.pickle"
with open(src_transforms, 'rb') as f:
    lst_transforms = pickle.load(f)
# data = gpd.read_file(src_gpkg)
# data['is_absurd'] = np.zeros(len(data), dtype=np.bool)
# lst_absurds = np.zeros(len(lst_transforms), dtype=np.bool)
lst_absurds = [False] * len(lst_transforms)
print(len(lst_absurds))

# loop over nodes:
for id_el, el in tqdm(enumerate(lst_transforms), total=len(lst_transforms)):
    # print(id_el)
    lvl_el = el[0]

    if lvl_el == 0:
        continue
    
    # compare to parent value
    val = get_translate_from_element(el)
    id_parent = id_el-1
    
    while(lst_transforms[id_parent][0] != lvl_el-1):
        id_parent -= 1

    parent_val = get_translate_from_element(lst_transforms[id_parent])
    diff = abs(parent_val - val)
    if diff > 10 or lst_absurds[id_parent] == True:
        lst_absurds[id_el] = True

print("Number of absurd values: ", np.sum(lst_absurds))
lst_transforms = [(*x,y) for x,y in zip(lst_transforms, lst_absurds)]
with open(src_results, 'wb') as f:
    pickle.dump(lst_transforms, f)

60053


100%|██████████| 60053/60053 [00:01<00:00, 48276.03it/s]


Number of absurd values:  22274


### Save results to GPKG

In [5]:
def export_points(data, columns, output_path, crs="EPSG:2056"):
    """
    Export list of [x, y, lvl, val1, ..., valn] to GPKG and SHP

    Parameters:
        data: list of lists
        output_path: base path without extension
        crs: coordinate reference system (default: Swiss LV95)
    """

    # Convert to DataFrame
    # n_extra = len(data[0]) - 3
    # columns = ["x", "y", "level"] + [f"val{i}" for i in range(1, n_extra + 1)]

    df = pd.DataFrame(data, columns=columns)

    # Create geometry
    geometry = [Point(xy) for xy in zip(df["x"], df["y"])]

    gdf = gpd.GeoDataFrame(df, geometry=geometry, crs=crs)

    # Save to GeoPackage
    # gpkg_path = output_path + ".gpkg"
    gdf.to_file(output_path, layer="displacements", driver="GPKG")

    # # Save to Shapefile
    # shp_path = output_path + ".shp"
    # gdf.to_file(shp_path, driver="ESRI Shapefile")

    print(f"Saved: {output_path}")
    
def export_points_and_bboxes(data, columns, bbox_data, output_path, offset, crs="EPSG:2056"):
    """
    Export points and bbox polygons as two layers in a single GPKG.

    Parameters:
        data: list of lists [x, y, lvl, val1, ..., valn]
        columns: column names for data
        bbox_data: list of dicts with keys 'min_bound', 'max_bound' and any attributes
        output_path: path with .gpkg extension
        crs: coordinate reference system (default: Swiss LV95)
    """
    from shapely.geometry import Point, Polygon

    # --- Layer 1: Points ---
    # apply offset
    df_points = pd.DataFrame(data, columns=columns)
    geometry_points = [Point(xy) for xy in zip(df_points["x"], df_points["y"])]
    gdf_points = gpd.GeoDataFrame(df_points, geometry=geometry_points, crs=crs)
    gdf_points.to_file(output_path, layer="displacements", driver="GPKG")

    # --- Layer 2: BBoxes ---
    rows = []
    for bbox in bbox_data:
        minx, miny = bbox["min_bound"][0] + offset[0], bbox["min_bound"][1] + offset[1]
        maxx, maxy = bbox["max_bound"][0] + offset[0], bbox["max_bound"][1] + offset[1]
        poly = Polygon([
            (minx, miny),
            (maxx, miny),
            (maxx, maxy),
            (minx, maxy),
            (minx, miny)
        ])
        row = {k: v for k, v in bbox.items() if k not in ("min_bound", "max_bound")}
        row["geometry"] = poly
        rows.append(poly)

    gdf_bboxes = gpd.GeoDataFrame(df_points, geometry=rows, crs=crs)
    gdf_bboxes.to_file(output_path, layer="tiles", driver="GPKG")  # append to same file

    print(f"Saved: {output_path}")
    print(f"  Points : {len(gdf_points)}")
    print(f"  BBoxes : {len(gdf_bboxes)}")


def compute_translation(bbox_dict, transform):
    xmin, ymin, zmin = bbox_dict['min_bound']
    xmax, ymax, zmax = bbox_dict['max_bound']
    center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
    translated = np.linalg.matmul(transform, center)
    norm = float(np.linalg.norm(translated - center))
    norm2d = float(np.linalg.norm(translated[0:2] - center[0:2]))
    # list_texts.append((cx, cy, norm, el_lvl))
    direction = (translated[0:2] - center[0:2]) / norm2d 

    return norm, direction

In [7]:
src_transforms = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_6_neigh_on_target\test_speed_up_to_lvl_8\results\pyramid_transforms_test_with_absurd_vals.pickle"
src_tile_original = r"D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_7_different_max_correspondances\down_to_0.2\swisssurface3d_2018_2589-1169_2056_5728.ply"
src_out_gpkg = os.path.join(os.path.dirname(src_transforms), 'points_translate.gpkg')

tile_original = o3d.io.read_point_cloud(src_tile_original)
z_mean = tile_original.compute_mean_and_covariance()[0][2]
offset = [2589500, 1169500, z_mean]
# tile_original.translate(np.array([-2589500, -1169500, 0]) - np.array([0, 0, z_mean]))

with open(src_transforms, 'rb') as f:
    lst_transforms = pickle.load(f)

data = []
bbox_data = []
for _, el in tqdm(enumerate(lst_transforms), total=len(lst_transforms)):
    el_lvl = el[0]
    bbox_dict = el[1][0]
    bbox_data.append(bbox_dict)
    bbox = o3d.geometry.AxisAlignedBoundingBox(
        min_bound=np.array(bbox_dict["min_bound"]),
        max_bound=np.array(bbox_dict["max_bound"])
    )

    transform = el[1][3]

    # translation:
    # cropped_tile = tile_original.crop(bbox)
    xmin, ymin, zmin = bbox_dict['min_bound']
    xmax, ymax, zmax = bbox_dict['max_bound']
    center = np.vstack([np.array([xmax + xmin, ymax + ymin, zmax + zmin]).reshape((3,1)) / 2, np.array([1])])
    # center = np.eye(4)
    translated = np.linalg.matmul(transform, center)
    norm = float(np.linalg.norm(translated - center))
    if norm == 0:
        print(xmin, ymin)
        print(transform)
        print('---')
    norm2d = float(np.linalg.norm(translated[0:2] - center[0:2]))
    direction = (translated[0:2] - center[0:2]) / norm2d 

    data.append((center[0][0] + offset[0], center[1][0] + offset[1], center[2][0] + offset[2], norm, direction[0][0], direction[1][0], el[0], el[-2], el[-1]))

columns = ['x', 'y', 'z', 'translation', 'translation_x', 'translation_y', 'lvl', 'is_leaf', 'absurd_status']

# Export all tiles
export_points_and_bboxes(
    data=data,
    bbox_data=bbox_data,
    columns=columns,
    output_path=src_out_gpkg,
    offset=offset,
)

# Export only leaves
mask_leaves = np.array([x[-2] for x in data], dtype=np.bool)
data_leaves = list(np.array(data)[mask_leaves])
bbox_data_leaves = list(np.array(bbox_data)[mask_leaves])

export_points_and_bboxes(
    data=data_leaves,
    bbox_data=bbox_data_leaves,
    columns=columns,
    output_path=src_out_gpkg.split('.gpkg')[0] + "_leaves.gpkg",
    offset=offset,
)

100%|██████████| 60053/60053 [00:01<00:00, 39948.74it/s]


Saved: D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_6_neigh_on_target\test_speed_up_to_lvl_8\results\points_translate.gpkg
  Points : 60053
  BBoxes : 60053
Saved: D:\GitHubProjects\Terranum_repo\pc_movement_tracking\data\test_6_neigh_on_target\test_speed_up_to_lvl_8\results\points_translate_leaves.gpkg
  Points : 45040
  BBoxes : 45040
